# TalentIQ - Training Pipeline Guide

This notebook explains everything that happens inside `train.py`.
It walks through the full ML pipeline from loading clean data to saving trained models and generating evaluation reports.

---

## Overview of the pipeline

The training pipeline has 6 main steps:

1. Load and split the data into train and test sets
2. Build a preprocessor that encodes and scales all features
3. Apply the preprocessor and then use SMOTE to balance the training data
4. Train three models with hyperparameter search
5. Evaluate all three models and compare them
6. Save the best model and generate reports

Each step is explained below.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

---

## Step 1 - Load and split the data

**What the code does in `load_and_prepare()`:**

First, the clean preprocessed data is loaded. Then `engineer_features()` is called to add the 7 new columns we created in the preprocessing notebook.

After that, the data is split into:
- `X` = all input features (everything except Attrition)
- `y` = the target column (Attrition: 0 or 1)

Then X and y are each split further into train and test sets using an 80/20 split.

**Why 80/20?**
We train on 80% of the data and test on the remaining 20%.
The test set is data the model has never seen. This gives us an honest evaluation of how the model would perform in the real world.
If we tested on training data, the model would look great but fail when deployed.

**Stratified split:**
We use `stratify=y` to make sure both the train and test sets have the same ratio of 0s and 1s.
Without stratification, random chance might put all the rare attrition cases into the test set,
which would make the model look worse than it is.

The split files are saved to `data/splits/` for traceability.

In [ ]:
from sklearn.model_selection import train_test_split

# Load data
try:
    df = pd.read_csv('../data/processed/processed_full.csv')
except FileNotFoundError:
    df = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
    df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

X = df.drop(columns=['Attrition'])
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Total rows: {len(df)}')
print(f'Train size: {len(X_train)} rows')
print(f'Test size: {len(X_test)} rows')
print(f'\nAttrition in train: {y_train.mean():.2%}')
print(f'Attrition in test: {y_test.mean():.2%}')
print('\nStratification check: both sets have approximately the same attrition rate.')

---

## Step 2 - Building the preprocessor

**What the code does in `build_preprocessor()`:**

Machine learning models cannot handle raw text. All features must be converted to numbers.
We use a `ColumnTransformer` from scikit-learn which applies different transformations to different columns at the same time.

There are three types of transformations applied:

### Ordinal Encoding
Used for columns where the categories have a meaningful order.

Example: `Education` has the order: Below College < College < Bachelor < Master < Doctor.
We encode this as 1, 2, 3, 4, 5.
The model can then understand that Doctor (5) is higher than Bachelor (3).

If we used one-hot encoding here, the model would treat each education level as completely unrelated to the others.
That would throw away the order information.

Ordinal columns: Education, EnvironmentSatisfaction, JobInvolvement, JobLevel, JobSatisfaction, PerformanceRating, RelationshipSatisfaction, StockOptionLevel, WorkLifeBalance.

### One-Hot Encoding
Used for columns where categories have NO meaningful order.

Example: `Department` has Sales, R&D, and HR.
There is no order between them. Sales is not greater than HR.
If we encoded them as 1, 2, 3, the model would think HR (3) is three times greater than Sales (1), which is wrong.

Instead, we create a separate binary column for each category.
We use `drop='first'` to avoid the dummy variable trap (one column is always predictable from the others).

One-hot columns: BusinessTravel, Department, EducationField, Gender, JobRole, MaritalStatus, OverTime.

### Standard Scaling
Used for all numerical columns.

The problem: MonthlyIncome ranges from 1,000 to 20,000. Age ranges from 18 to 60.
If we feed both raw to the model, MonthlyIncome values are hundreds of times larger than Age values.
Some models (like Logistic Regression) are sensitive to this and will unfairly weight MonthlyIncome more.

Standard scaling transforms each column to have mean=0 and standard deviation=1.
After scaling, both MonthlyIncome and Age are on the same relative scale.

This is applied to both original numerical features and the 7 engineered features.

In [ ]:
# Demonstrate Standard Scaling clearly
from sklearn.preprocessing import StandardScaler

# Example with two columns on very different scales
sample = pd.DataFrame({
    'Age': [25, 35, 45, 55],
    'MonthlyIncome': [2000, 5000, 10000, 20000]
})

scaler = StandardScaler()
scaled = scaler.fit_transform(sample)

print('Before scaling:')
print(sample)
print('\nAfter scaling (both columns are now on the same scale):')
print(pd.DataFrame(scaled, columns=['Age', 'MonthlyIncome']).round(3))

---

## Step 3 - Applying SMOTE to balance training data

**What the code does in `transform_and_resample()`:**

After encoding and scaling, the preprocessor is saved to `artifacts/preprocessor.pkl`.
This is important. We need to reuse the exact same preprocessor (fitted on training data) when making predictions later.
If we fit a new preprocessor on test data, it would see different scaling parameters and the predictions would be wrong.

Then SMOTE is applied.

**What is SMOTE?**
SMOTE stands for Synthetic Minority Over-sampling Technique.

The problem: We only have about 237 employees who left out of 1176 in training (16%). This imbalance can cause the model to mostly predict "No attrition" and be 84% accurate while being completely useless at identifying who will leave.

SMOTE fixes this by creating synthetic (artificial) new training examples of the minority class.
It does this by taking an existing minority example and generating a new one that is slightly different, based on the nearest neighbors in feature space.

This is different from just duplicating rows. The new examples are generated from the patterns in existing minority examples, so the model gets a richer view of what attrition looks like.

Important: SMOTE is applied ONLY to training data. The test set remains unchanged so our evaluation is realistic.

In [ ]:
# Demonstrate the effect of SMOTE conceptually
print('Before SMOTE (training set):')
print(f'  Total rows: {len(y_train)}')
print(f'  Class 0 (Stayed): {(y_train == 0).sum()}')
print(f'  Class 1 (Left): {(y_train == 1).sum()}')
print(f'  Class 1 percentage: {y_train.mean():.2%}')

print('\nAfter SMOTE (training set - simulated):')
stayed = (y_train == 0).sum()
print(f'  Class 0 (Stayed): {stayed}')
print(f'  Class 1 (Left): {stayed} (matched to majority class)')
print(f'  New total rows: ~{stayed * 2}')
print(f'  Class 1 percentage: ~50%')

print('\nTest set (unchanged by SMOTE):')
print(f'  Class 0: {(y_test == 0).sum()}')
print(f'  Class 1: {(y_test == 1).sum()}')

---

## Step 4 - Training models with hyperparameter search

**What the code does in `search_model()` and `train_all()`:**

Three models are trained in this project. Each one uses a different approach to classification.

### Model 1 - Logistic Regression

Logistic Regression is the simplest classifier and acts as our baseline.
It finds a linear decision boundary that separates employees who will leave from those who will stay.

Think of it as drawing a straight line through the data to separate the two groups.
It works well when the relationship between features and the target is approximately linear.

Parameter tuned:
- `C`: controls how much the model is penalized for complexity. Lower C = stronger regularization (simpler model). Higher C = model fits training data more closely.
- `class_weight='balanced'`: adjusts for remaining imbalance by giving more weight to the minority class.

### Model 2 - Random Forest

Random Forest builds many decision trees, each on a random subset of the data and features.
Each tree makes its own prediction and the final answer is the majority vote across all trees.

Why is this better than a single tree?
A single decision tree tends to memorize the training data (overfitting).
By averaging many trees that each saw different parts of the data, the errors cancel out and the result is more reliable.

Parameters tuned:
- `n_estimators`: how many trees to build
- `max_depth`: how deep each tree can grow (deeper = more complex, risk of overfitting)
- `min_samples_split` and `min_samples_leaf`: control how much data a node needs before it splits further

### Model 3 - XGBoost

XGBoost is a gradient boosting model. Instead of building trees independently like Random Forest, it builds them sequentially.
Each new tree focuses on correcting the mistakes of the previous trees.

This makes XGBoost very powerful for structured tabular data like this HR dataset.
It typically outperforms Random Forest and Logistic Regression on imbalanced classification problems.

XGBoost also has `scale_pos_weight=5.25` set directly.
This is the ratio of negative to positive samples (stayed/left = 1233/237 ≈ 5.2).
It tells XGBoost to penalize false negatives (missing an employee who will leave) 5x more heavily.

Parameters tuned:
- `n_estimators`: number of boosting rounds
- `learning_rate`: how much each new tree corrects the previous errors (lower = more careful but slower)
- `max_depth`: depth of each tree
- `subsample` and `colsample_bytree`: fraction of data and features used per tree (helps prevent overfitting)

---

### Hyperparameter Search

Every model has parameters that must be set before training (hyperparameters).
We do not guess these. We search for the best combination.

Two search strategies are used:

**GridSearchCV** (used for Logistic Regression):
Tries every possible combination of the specified values.
Good when the search space is small (e.g., 2 values of C x 1 penalty = 2 combinations).

**RandomizedSearchCV** (used for Random Forest and XGBoost):
Randomly samples n_iter combinations from the parameter space.
Used when the search space is large. Trying all combinations would take too long.
We set n_iter=5 to keep training time manageable.

Both use **3-fold cross-validation**.
For each parameter combination, the training data is split into 3 parts.
The model trains on 2 parts and validates on the 3rd, rotating which part is held out.
The average score across 3 folds is used to rank parameter combinations.

The scoring metric used is `f1_macro` because the data is imbalanced.

In [ ]:
# Show the hyperparameter configuration from hyperparameters.yaml
import yaml
with open('../config/hyperparameters.yaml', 'r') as f:
    hp = yaml.safe_load(f)

for model_name, config in hp.items():
    print(f'--- {model_name} ---')
    print(f'Search type: {config["search"]}')
    print(f'CV folds: {config["cv"]}')
    print(f'Scoring: {config["scoring"]}')
    if 'param_grid' in config:
        print(f'Param grid: {config["param_grid"]}')
    if 'param_distributions' in config:
        print(f'Param distributions: {config["param_distributions"]}')
        print(f'n_iter: {config["n_iter"]}')
    print()

---

## Step 5 - Evaluating models

**What the code does in `evaluate_model()` in `metrics.py`:**

After training each model, we run it on the held-out test set and calculate several metrics.
Each metric tells us something different about how well the model is doing.

### Accuracy
The percentage of predictions that were correct.
Example: 85% accuracy means 85 out of 100 employees were correctly classified.

**Why accuracy alone is not enough here:**
If 84% of employees stayed, a model that always predicts "stay" would be 84% accurate without learning anything.
That is why we use F1-macro as the primary metric.

### F1-macro
F1 score is the harmonic mean of precision and recall.
Macro means we calculate F1 for each class separately and then take the average.
This gives equal weight to both the majority (stayed) and minority (left) class.
A model that ignores the minority class will get a low F1-macro.

### Precision
Of all the employees the model predicted would leave, how many actually left?
A low precision means the model is sending HR to check on employees who are actually fine (wasted effort).

### Recall
Of all the employees who actually left, how many did the model correctly identify?
A low recall means the model is missing at-risk employees (costly oversight).

### ROC-AUC
This measures how well the model separates the two classes across all possible thresholds.
An AUC of 0.5 means random guessing. An AUC of 1.0 means perfect separation.
It is used as a tiebreaker when two models have the same F1-macro.

### FPR and FNR
**FPR (False Positive Rate):** Employees who stayed but were predicted to leave. Wasted HR interviews.
**FNR (False Negative Rate):** Employees who left but were predicted to stay. Missed good people we could have retained.

Both matter in recruitment context. The misclassification analysis section in the summary report shows which model balances these best.

In [ ]:
# Show what the confusion matrix means in this context
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(6, 5))
confusion = np.array([[200, 30], [15, 50]])
im = ax.imshow(confusion, cmap='Blues')

labels = [['True Negative\n(Predicted Stay,\nActually Stayed)',
           'False Positive\n(Predicted Left,\nActually Stayed)\nWasted interview'],
          ['False Negative\n(Predicted Stay,\nActually Left)\nMissed talent',
           'True Positive\n(Predicted Left,\nActually Left)']]

for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{confusion[i,j]}\n{labels[i][j]}',
                ha='center', va='center', fontsize=8,
                color='black' if confusion[i,j] < 150 else 'white')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted Stay', 'Predicted Left'])
ax.set_yticklabels(['Actually Stayed', 'Actually Left'])
ax.set_title('Confusion Matrix - What each cell means')
plt.tight_layout()
plt.show()

---

## Step 6 - Saving models, plots, and reports

**What the code does after each model is evaluated:**

### Saving models
Each trained model is saved to `artifacts/models/` as a `.pkl` file using `joblib`.
This means we do not have to retrain every time we want to make a prediction.
The inference script loads the saved model and runs it directly.

The preprocessor is also saved to `artifacts/preprocessor.pkl`.
When making a prediction on new data, we must apply the same preprocessing transformations.
We load the saved preprocessor and use it, rather than fitting a new one.

### Plots generated per model
- **Confusion Matrix:** Shows TP, TN, FP, FN counts visually.
- **ROC Curve:** Shows how the true positive rate changes vs false positive rate at different thresholds.
- **Feature Importance:** Shows which features the model relied on most when making predictions.

### Model comparison report
After all three models are trained, `save_metrics()` writes a CSV with metrics for each model.
`plot_model_comparison()` creates a bar chart comparing F1-macro, Precision, Recall, and AUC across all three models.
`save_summary()` writes a Markdown report to `reports/summary.md`.

The best model is selected by highest F1-macro, with ROC-AUC as a tiebreaker.
The summary explains the business impact of each error type and justifies the model selection.

In [ ]:
# Load and display the metrics if they exist
import os

metrics_path = '../reports/metrics/metrics.csv'
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    print('Model Comparison Results:')
    print(metrics_df.to_string(index=False))
    
    best = metrics_df.sort_values(['F1-macro', 'ROC-AUC'], ascending=False).iloc[0]
    print(f'\nBest model: {best["Model"]}')
    print(f'F1-macro: {best["F1-macro"]} | ROC-AUC: {best["ROC-AUC"]}')
else:
    print('metrics.csv not found. Run the training pipeline first to generate results.')
    print('Expected at:', metrics_path)

In [ ]:
# Show model comparison chart if metrics exist
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    plot_cols = ['Accuracy', 'F1-macro', 'Precision', 'Recall', 'ROC-AUC']
    
    x = np.arange(len(metrics_df))
    width = 0.15
    
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, col in enumerate(plot_cols):
        ax.bar(x + i * width, metrics_df[col], width, label=col)
    
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(metrics_df['Model'])
    ax.set_ylim(0, 1.1)
    ax.set_title('Model Comparison')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Run the training pipeline to generate metrics and see this chart.')

---

## The full pipeline in one view

Here is the complete flow from raw data to saved model:

```
Raw CSV
  |
  v
load_data()
  - Drop useless columns
  - Map Attrition to 0/1
  - Convert ordinal integers to text
  |
  v
validate_data()
  - Check all required columns exist
  |
  v
handle_missing_values()
  - Numerical: fill with median
  - Categorical: fill with mode
  |
  v
remove_duplicates()
  |
  v
handle_outliers()
  - IQR clipping on 4 columns
  |
  v
engineer_features()
  - Add 7 new columns
  |
  v
train_test_split() - 80/20, stratified
  |
  v
ColumnTransformer
  - OrdinalEncoder for ranked categories
  - OneHotEncoder for unranked categories
  - StandardScaler for all numerical features
  |
  v
SMOTE - balance training data
  |
  v
Train 3 models with cross-validated hyperparameter search
  - Logistic Regression (GridSearchCV)
  - Random Forest (RandomizedSearchCV)
  - XGBoost (RandomizedSearchCV)
  |
  v
Evaluate on test set
  - Accuracy, F1-macro, Precision, Recall, ROC-AUC, FPR, FNR
  |
  v
Save artifacts
  - models to artifacts/models/
  - preprocessor to artifacts/preprocessor.pkl
  - plots to reports/figures/
  - metrics to reports/metrics/
  - summary to reports/summary.md
```

---

## Key design decisions and why

| Decision | Why |
|---|---|
| F1-macro as primary metric | Data is imbalanced. Accuracy is misleading. F1-macro gives equal weight to both classes. |
| SMOTE on training only | We must not leak synthetic data into evaluation. Test set must reflect the real world. |
| Preprocessor saved separately | Inference must use the same transformations. Fitting a new preprocessor on new data would give different scaling. |
| Config-driven paths and settings | Changing a path or mode requires editing YAML, not Python code. Reduces bugs. |
| lru_cache on config loaders | YAML files are read only once per run. Calling `load_config()` 10 times does not read the file 10 times. |
| scale_pos_weight in XGBoost | Directly tells XGBoost the class ratio so it learns to treat false negatives as more costly. |
| 3-fold CV instead of 5 | Faster training. Good enough for a dataset of this size. |
| stratify=y in train_test_split | Ensures both sets have the same class distribution. Prevents biased evaluation. |